# BirdCLEF 2026 — BirdNET Inference Notebook
Auto-generated by the autonomous BirdNET embedding agent (val macro_auc_ge3 = 0.551056).


In [ ]:
import subprocess, shutil, os, sys
import numpy as np, pandas as pd, librosa, tensorflow as tf
from pathlib import Path

def _find_files(root, ext):
    found = []
    for dirpath, _, files in os.walk(root, followlinks=True):
        for f in files:
            if f.endswith(ext):
                found.append(os.path.join(dirpath, f))
    return found

# --- 1. Install wheels (deduplicated by filename) ---
_all_wheels = _find_files('/kaggle/input', '.whl')
_dedup = {}
for w in _all_wheels:
    name = Path(w).name
    if name not in _dedup:
        _dedup[name] = w
_wheels_to_install = list(_dedup.values())
print(f'Found wheels: {list(_dedup.keys())}')

for w in _wheels_to_install:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '-q', w],
        capture_output=True, text=True
    )
    status = 'OK' if r.returncode == 0 else f'FAILED: {r.stderr[-200:]}'
    print(f'  {Path(w).name}: {status}')

# verify
try:
    import importlib
    importlib.import_module('birdnet')
    print('birdnet import: OK')
except ImportError as e:
    raise RuntimeError(f'birdnet still not importable after install: {e}')

# --- 2. Copy TFLite model cache so birdnet finds it without downloading ---
_tflite = _find_files('/kaggle/input', '.tflite')
if _tflite:
    _tflite_src = Path(_tflite[0])
    _bn_dst = Path.home() / '.local' / 'share' / 'birdnet' / 'acoustic-models' / 'v2.4' / 'tf'
    _bn_dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(_tflite_src), str(_bn_dst / 'model-fp32.tflite'))
    _labels_src = _tflite_src.parent / 'labels'
    if _labels_src.exists():
        shutil.copytree(str(_labels_src), str(_bn_dst / 'labels'), dirs_exist_ok=True)
        print(f'BirdNET model cache ready ({len(list(_labels_src.iterdir()))} label files)')
else:
    raise RuntimeError('No .tflite file found in /kaggle/input')

print('Libraries loaded.')

In [ ]:
# Auto-discover competition data and model (uses _find_files from cell 1)
_sample_subs = _find_files('/kaggle/input', 'sample_submission.csv')
if not _sample_subs:
    raise FileNotFoundError('sample_submission.csv not found — is BirdCLEF+ 2026 attached?')
COMP_DIR = str(Path(_sample_subs[0]).parent)
print(f'COMP_DIR: {COMP_DIR}')

_keras_files = _find_files('/kaggle/input', '.keras')
_npz_files   = _find_files('/kaggle/input', '.npz')
if not _keras_files:
    raise FileNotFoundError('model.keras not found')
MODEL_DIR = str(Path(_keras_files[0]).parent)
print(f'MODEL_DIR: {MODEL_DIR}')

SR_LOAD    = 32000
BIRDNET_SR = 48000
CLIP_SEC   = 5.0
CHUNK_SEC  = 3.0
BATCH      = 32

In [ ]:
import birdnet
print('Loading BirdNET v2.4 ...')
bnet = birdnet.load('acoustic', '2.4', 'tf')

# Patch model.keras: strip quantization_config from Dense configs (Keras version mismatch)
import zipfile, json, tempfile

_src = os.path.join(MODEL_DIR, 'model.keras')
_tmp = tempfile.mktemp(suffix='.keras')

def _strip_quant(obj):
    if isinstance(obj, dict):
        return {k: _strip_quant(v) for k, v in obj.items() if k != 'quantization_config'}
    if isinstance(obj, list):
        return [_strip_quant(i) for i in obj]
    return obj

with zipfile.ZipFile(_src, 'r') as zin, zipfile.ZipFile(_tmp, 'w', zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == 'config.json':
            cfg = json.loads(data.decode('utf-8'))
            cfg = _strip_quant(cfg)
            data = json.dumps(cfg).encode('utf-8')
        zout.writestr(item, data)

model = tf.keras.models.load_model(_tmp, compile=False)
os.remove(_tmp)
print('model loaded OK')

sc       = np.load(os.path.join(MODEL_DIR, 'scaler.npz'))
sc_mean  = sc['mean'].astype(np.float32)
sc_scale = sc['scale'].astype(np.float32)

sample_sub   = pd.read_csv(os.path.join(COMP_DIR, 'sample_submission.csv'))
species_cols = [c for c in sample_sub.columns if c != 'row_id']
print(f'Ready. Species: {len(species_cols)}')

In [ ]:
def _prepare_chunk(y, sr):
    y = np.asarray(y, dtype=np.float32).reshape(-1)
    if sr != BIRDNET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=BIRDNET_SR)
    keep = int(CLIP_SEC * BIRDNET_SR)
    y = y[:keep] if len(y) > keep else y
    need = int(CHUNK_SEC * BIRDNET_SR)
    s0 = max(0, (len(y) - need) // 2)
    chunk = y[s0:s0+need]
    return np.pad(chunk, (0, max(0, need - len(chunk)))).astype(np.float32)

def _embed_predict(audio_list, session):
    inputs = [(np.ascontiguousarray(_prepare_chunk(y, SR_LOAD), np.float32), BIRDNET_SR)
              for y in audio_list]
    res  = session.run_arrays(inputs)
    embs = np.asarray(res.embeddings[:, 0, :], np.float32)
    embs = (embs - sc_mean) / sc_scale
    return model.predict(embs, verbose=0).astype(np.float32)

In [ ]:
test_dir   = Path(COMP_DIR) / 'test_soundscapes'
test_files = sorted(test_dir.glob('*.ogg'))
print(f'test_dir: {test_dir}  exists: {test_dir.exists()}')
print(f'Test soundscapes: {len(test_files)}')
if test_files:
    print(f'  first: {test_files[0]}')

clip_n    = int(CLIP_SEC * SR_LOAD)
rows      = []
audio_buf = []
row_ids   = []

if len(test_files) == 0:
    print('WARNING: no test files found — submission will be uniform')
else:
    try:
        with bnet.encode_session(batch_size=BATCH, prefetch_ratio=2, n_workers=4, n_producers=2) as sess:
            def flush():
                if not audio_buf: return
                preds = _embed_predict(audio_buf, sess)
                for rid, pred in zip(row_ids, preds):
                    r = {'row_id': rid}
                    r.update(zip(species_cols, pred.tolist()))
                    rows.append(r)
                audio_buf.clear(); row_ids.clear()

            for fpath in test_files:
                stem   = fpath.stem
                y_full, _ = librosa.load(str(fpath), sr=SR_LOAD, mono=True)
                n_win  = max(1, int(np.ceil(len(y_full) / clip_n)))
                for w in range(n_win):
                    seg = y_full[w*clip_n:(w+1)*clip_n]
                    if len(seg) < clip_n:
                        seg = np.pad(seg, (0, clip_n - len(seg)))
                    audio_buf.append(seg.astype(np.float32))
                    row_ids.append(f'{stem}_{(w+1)*int(CLIP_SEC)}')
                    if len(audio_buf) >= BATCH:
                        flush()
            flush()
    except Exception as e:
        print(f'ERROR in encode_session: {e}')

print(f'Predicted {len(rows)} windows')

In [ ]:
if rows:
    sub = pd.DataFrame(rows)
    sub = sample_sub[['row_id']].merge(sub, on='row_id', how='left')
else:
    print('WARNING: no predictions — uniform submission')
    sub = sample_sub[['row_id']].copy()
    for col in species_cols:
        sub[col] = np.nan

sub[species_cols] = sub[species_cols].fillna(1.0 / len(species_cols)).clip(0.0, 1.0)
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Submission saved: {sub.shape}')
sub.head(3)